In [10]:
# General
import numpy as np
import pandas as pd 
import os
import bs4
import requests
import warnings
import re
import html
import unidecode

In [11]:
df_initial = pd.read_csv("Covid_Dataset_ver_6.csv")
df_initial

,Code,continent,country,Day,total_cases,new_cases,new_cases_smoothed,total_deaths,new_deaths,new_deaths_smoothed,...,Nucleocapsid (N-protein) Multiple groups,Nucleocapsid (N-protein) Seniors (65+ years),"Nucleocapsid (N-protein), Spike Adults (18-64 years)","Nucleocapsid (N-protein), Spike Children and Youth (0-17 years)","Nucleocapsid (N-protein), Spike Multiple groups","Nucleocapsid (N-protein), Spike Seniors (65+ years)",Spike Adults (18-64 years),Spike Children and Youth (0-17 years),Spike Multiple groups,Spike Seniors (65+ years)
0,AFG,Asia,Afghanistan,2020-02-24,5.0,5.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AFG,Asia,Afghanistan,2020-02-25,5.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AFG,Asia,Afghanistan,2020-02-26,5.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AFG,Asia,Afghanistan,2020-02-27,5.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AFG,Asia,Afghanistan,2020-02-28,5.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
254395,ZWE,Africa,Zimbabwe,2023-01-28,261606.0,0.0,232.143,5652.0,0.0,2.143,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
254396,ZWE,Africa,Zimbabwe,2023-01-29,261606.0,0.0,232.143,5652.0,0.0,2.143,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
254397,ZWE,Africa,Zimbabwe,2023-01-30,261606.0,0.0,232.143,5652.0,0.0,2.143,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
254398,ZWE,Africa,Zimbabwe,2023-01-31,261606.0,0.0,0.000,5652.0,0.0,0.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
df_initial['Day'] = pd.to_datetime(df_initial['Day'])

df_initial.dtypes

Code                                                           object
continent                                                      object
country                                                        object
Day                                                    datetime64[ns]
total_cases                                                   float64
                                                            ...      
Nucleocapsid (N-protein), Spike Seniors (65+ years)           float64
Spike Adults (18-64 years)                                    float64
Spike Children and Youth (0-17 years)                         float64
Spike Multiple groups                                         float64
Spike Seniors (65+ years)                                     float64
Length: 106, dtype: object

In [13]:
temp_df = df_initial[df_initial["country"] == 'United States']
temp_df

,Code,continent,country,Day,total_cases,new_cases,new_cases_smoothed,total_deaths,new_deaths,new_deaths_smoothed,...,Nucleocapsid (N-protein) Multiple groups,Nucleocapsid (N-protein) Seniors (65+ years),"Nucleocapsid (N-protein), Spike Adults (18-64 years)","Nucleocapsid (N-protein), Spike Children and Youth (0-17 years)","Nucleocapsid (N-protein), Spike Multiple groups","Nucleocapsid (N-protein), Spike Seniors (65+ years)",Spike Adults (18-64 years),Spike Children and Youth (0-17 years),Spike Multiple groups,Spike Seniors (65+ years)
239110,USA,North America,United States,2020-01-22,1.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
239111,USA,North America,United States,2020-01-23,1.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
239112,USA,North America,United States,2020-01-24,2.0,1.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
239113,USA,North America,United States,2020-01-25,2.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
239114,USA,North America,United States,2020-01-26,5.0,3.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
240212,USA,North America,United States,2023-01-28,102272030.0,4436.0,39777.143,1107493.0,11.0,490.429,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
240213,USA,North America,United States,2023-01-29,102274637.0,2607.0,39696.571,1107493.0,0.0,490.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
240214,USA,North America,United States,2023-01-30,102301714.0,27077.0,39177.571,1107703.0,210.0,476.571,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
240215,USA,North America,United States,2023-01-31,102336754.0,35040.0,36262.286,1108330.0,627.0,453.143,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
df_gisaid_exp_f1 = pd.read_csv("df_gisaid_exp_f1.csv")

df_gisaid_exp_f1 = df_gisaid_exp_f1.rename(columns={'location' : 'country',
                                                    'week' : 'Day'})

df_gisaid_exp_f1['Day'] = pd.to_datetime(df_gisaid_exp_f1['Day'])

df_gisaid_exp_f1

,country,Day,n
0,Afghanistan,2019-12-29,0
1,Afghanistan,2020-01-05,0
2,Afghanistan,2020-01-12,0
3,Afghanistan,2020-01-19,0
4,Afghanistan,2020-01-26,0
...,...,...,...
37057,Zimbabwe,2023-03-26,0
37058,Zimbabwe,2023-04-02,0
37059,Zimbabwe,2023-04-09,0
37060,Zimbabwe,2023-04-16,0


In [15]:
df_gisaid_exp_f2 = pd.read_csv("df_gisaid_exp_f2.csv")

df_gisaid_exp_f2 = df_gisaid_exp_f2.rename(columns={'location' : 'country',
                                                    'week' : 'Day'})

df_gisaid_exp_f2['Day'] = pd.to_datetime(df_gisaid_exp_f2['Day'])

df_gisaid_exp_f2

,country,Day,Alpha*,Beta,Delta*,Epsilon,Eta,Gamma,Iota,Kappa,...,Omicron BA.2*,Omicron BA.2.75*,Omicron BA.4*,Omicron BA.5*,Omicron BQ.*,Recombinants,XBB*,other,wildtype A*,wildtype B*
0,Afghanistan,2019-12-29,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Afghanistan,2020-01-05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Afghanistan,2020-01-12,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Afghanistan,2020-01-19,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Afghanistan,2020-01-26,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33577,Zimbabwe,2023-03-26,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
33578,Zimbabwe,2023-04-02,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
33579,Zimbabwe,2023-04-09,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
33580,Zimbabwe,2023-04-16,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [16]:
df_diversity = pd.read_excel("diversity_df_new.xlsx")

df_diversity = df_diversity.rename(columns={'location' : 'country',
                                            'week' : 'Day'})

df_diversity['Day'] = pd.to_datetime(df_diversity['Day'])

df_diversity = df_diversity.drop("n_seq_perwk", axis=1)

df_diversity = df_diversity.replace({'USA' : 'United States'})

df_diversity

,country,Day,weekly_cases,sequenc_eff,richness,diversity
0,Afghanistan,2019-12-29,0,0.0,NaN,NaN
1,Afghanistan,2020-01-05,0,0.0,NaN,NaN
2,Afghanistan,2020-01-12,0,0.0,NaN,NaN
3,Afghanistan,2020-01-19,0,0.0,NaN,NaN
4,Afghanistan,2020-01-26,0,0.0,NaN,NaN
...,...,...,...,...,...,...
33577,Zimbabwe,2023-03-26,41,0.0,NaN,NaN
33578,Zimbabwe,2023-04-02,29,0.0,NaN,NaN
33579,Zimbabwe,2023-04-09,29,0.0,NaN,NaN
33580,Zimbabwe,2023-04-16,25,0.0,NaN,NaN


In [17]:
temp_df = df_diversity[df_diversity["country"] == 'United States']
temp_df

,country,Day,weekly_cases,sequenc_eff,richness,diversity
32190,United States,2019-12-29,0,inf,1.0,1.000000
32191,United States,2020-01-05,0,0.00,NaN,NaN
32192,United States,2020-01-12,0,inf,1.0,1.000000
32193,United States,2020-01-19,6,2.17,3.0,2.522194
32194,United States,2020-01-26,2,4.50,3.0,2.336574
...,...,...,...,...,...,...
32359,United States,2023-03-26,176358,0.03,140.0,9.274340
32360,United States,2023-04-02,0,inf,134.0,9.351004
32361,United States,2023-04-09,103472,0.02,90.0,10.266770
32362,United States,2023-04-16,104057,0.00,50.0,11.233580


## Merge

In [18]:
df_ver_7 = df_initial.copy()

df_ver_7 = df_ver_7.merge(df_gisaid_exp_f2, on=["country", "Day"], how='left')

df_ver_7

,Code,continent,country,Day,total_cases,new_cases,new_cases_smoothed,total_deaths,new_deaths,new_deaths_smoothed,...,Omicron BA.2*,Omicron BA.2.75*,Omicron BA.4*,Omicron BA.5*,Omicron BQ.*,Recombinants,XBB*,other,wildtype A*,wildtype B*
0,AFG,Asia,Afghanistan,2020-02-24,5.0,5.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AFG,Asia,Afghanistan,2020-02-25,5.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AFG,Asia,Afghanistan,2020-02-26,5.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AFG,Asia,Afghanistan,2020-02-27,5.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AFG,Asia,Afghanistan,2020-02-28,5.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
254395,ZWE,Africa,Zimbabwe,2023-01-28,261606.0,0.0,232.143,5652.0,0.0,2.143,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
254396,ZWE,Africa,Zimbabwe,2023-01-29,261606.0,0.0,232.143,5652.0,0.0,2.143,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
254397,ZWE,Africa,Zimbabwe,2023-01-30,261606.0,0.0,232.143,5652.0,0.0,2.143,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
254398,ZWE,Africa,Zimbabwe,2023-01-31,261606.0,0.0,0.000,5652.0,0.0,0.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
df_ver_8 = df_ver_7.copy()

df_ver_8 = df_ver_8.merge(df_gisaid_exp_f1, on=["country", "Day"], how='left')

df_ver_8

,Code,continent,country,Day,total_cases,new_cases,new_cases_smoothed,total_deaths,new_deaths,new_deaths_smoothed,...,Omicron BA.2.75*,Omicron BA.4*,Omicron BA.5*,Omicron BQ.*,Recombinants,XBB*,other,wildtype A*,wildtype B*,n
0,AFG,Asia,Afghanistan,2020-02-24,5.0,5.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AFG,Asia,Afghanistan,2020-02-25,5.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AFG,Asia,Afghanistan,2020-02-26,5.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AFG,Asia,Afghanistan,2020-02-27,5.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AFG,Asia,Afghanistan,2020-02-28,5.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
254395,ZWE,Africa,Zimbabwe,2023-01-28,261606.0,0.0,232.143,5652.0,0.0,2.143,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
254396,ZWE,Africa,Zimbabwe,2023-01-29,261606.0,0.0,232.143,5652.0,0.0,2.143,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
254397,ZWE,Africa,Zimbabwe,2023-01-30,261606.0,0.0,232.143,5652.0,0.0,2.143,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
254398,ZWE,Africa,Zimbabwe,2023-01-31,261606.0,0.0,0.000,5652.0,0.0,0.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
df_ver_9 = df_ver_8.copy()

df_ver_9 = df_ver_9.merge(df_diversity, on=["country", "Day"], how='left')

df_ver_9

,Code,continent,country,Day,total_cases,new_cases,new_cases_smoothed,total_deaths,new_deaths,new_deaths_smoothed,...,Recombinants,XBB*,other,wildtype A*,wildtype B*,n,weekly_cases,sequenc_eff,richness,diversity
0,AFG,Asia,Afghanistan,2020-02-24,5.0,5.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AFG,Asia,Afghanistan,2020-02-25,5.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AFG,Asia,Afghanistan,2020-02-26,5.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AFG,Asia,Afghanistan,2020-02-27,5.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AFG,Asia,Afghanistan,2020-02-28,5.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
254395,ZWE,Africa,Zimbabwe,2023-01-28,261606.0,0.0,232.143,5652.0,0.0,2.143,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
254396,ZWE,Africa,Zimbabwe,2023-01-29,261606.0,0.0,232.143,5652.0,0.0,2.143,...,0.0,0.0,0.0,0.0,0.0,0.0,784.0,0.0,NaN,NaN
254397,ZWE,Africa,Zimbabwe,2023-01-30,261606.0,0.0,232.143,5652.0,0.0,2.143,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
254398,ZWE,Africa,Zimbabwe,2023-01-31,261606.0,0.0,0.000,5652.0,0.0,0.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
df_ver_9.to_csv("df_ver_9.csv", index=False)